In [ ]:
import numpy as np
import pandas as pd

In [ ]:
file_path = "processing_data_4.xlsx"
df = pd.read_excel(file_path)

In [3]:
# Chuẩn hoá kiểu dữ liệu & trim khoảng trắng ở text
text_cols = ["Category", "Title", "Brand", "Currency", "PriceSegment",
            "PopularitySegment", "RatingSegment", "BrandCategory", "DuplicateStatus"]
for c in text_cols:
    if c in df_clean.columns:
        df_clean[c] = df_clean[c].astype(str).str.strip()

numeric_cols = ["Price", "RatingAverage", "QuantitySold", "FeaturePCA1", "FeaturePCA2"]
for c in numeric_cols:
    if c in df_clean.columns:
        df_clean[c] = pd.to_numeric(df_clean[c], errors="coerce")

NameError: name 'df_clean' is not defined

In [ ]:
# Loại giá trị không hợp lệ & ràng buộc tối thiểu
#    - Price phải >0, RatingAverage trong (0,5], QuantitySold >=0
df_clean.loc[df_clean["Price"] <= 0, "Price"] = np.nan
df_clean.loc[(df_clean["RatingAverage"] <= 0) | (df_clean["RatingAverage"] > 5), "RatingAverage"] = np.nan
df_clean.loc[df_clean["QuantitySold"] < 0, "QuantitySold"] = np.nan

# Bỏ các dòng thiếu bắt buộc lần nữa sau khi ràng buộc
df_clean = df_clean.dropna(subset=["ProductID", "Title", "Brand", "Price"]).reset_index(drop=True)

In [ ]:
# Chuẩn hoá Currency (nếu không phải VND thì gắn nhãn Unknown để bạn dễ lọc)
if "Currency" in df_clean.columns:
    df_clean["Currency"] = df_clean["Currency"].str.upper()
    df_clean.loc[~df_clean["Currency"].isin(["VND", "VNĐ"]), "Currency"] = "UNKNOWN"

In [ ]:
# Impute thông minh cho RatingAverage còn thiếu: theo Brand → theo Category → median toàn cục
if "RatingAverage" in df_clean.columns:
    rat = df_clean["RatingAverage"].copy()

    # theo Brand
    if "Brand" in df_clean.columns:
        rat = rat.fillna(df_clean.groupby("Brand")["RatingAverage"].transform("median"))

    # theo Category
    if "Category" in df_clean.columns:
        rat = rat.fillna(df_clean.groupby("Category")["RatingAverage"].transform("median"))

    # median toàn cục
    rat = rat.fillna(rat.median(skipna=True))
    df_clean["RatingAverage"] = rat.round(2)

In [ ]:
# Đồng bộ lại RatingSegment từ RatingAverage (thêm nhãn Unknown cho ô còn NaN)
bins = [0, 3, 4, 5]
labels = ["Poor", "Good", "Excellent"]
rating_cut = pd.cut(df_clean["RatingAverage"], bins=bins, labels=labels, include_lowest=True)
if hasattr(rating_cut, "cat"):
    rating_cut = rating_cut.cat.add_categories("Unknown")
df_clean["RatingSegment"] = rating_cut.fillna("Unknown")

In [ ]:
# Xử lý outlier (winsorize/clip) cho Price & QuantitySold bằng IQR
def clip_iqr(s: pd.Series, k: float = 1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    low, high = q1 - k*iqr, q3 + k*iqr
    return s.clip(lower=max(0, low), upper=high)

if "Price" in df_clean.columns:
    df_clean["Price"] = clip_iqr(df_clean["Price"], k=1.5)

if "QuantitySold" in df_clean.columns:
    df_clean["QuantitySold"] = clip_iqr(df_clean["QuantitySold"], k=1.5)

In [201]:
# Tạo các biến hỗ trợ phân tích (không thay thế biến gốc)
if "Price" in df_clean.columns:
    df_clean["PriceLog1p"] = np.log1p(df_clean["Price"])
if "QuantitySold" in df_clean.columns:
    df_clean["QuantitySoldLog1p"] = np.log1p(df_clean["QuantitySold"].fillna(0))

In [202]:
df_clean = df_clean.drop_duplicates(subset=["ProductID"], keep="first")

In [203]:
# Kiểm tra logic nhãn (ví dụ: Apple/Samsung/OPPO không rơi vào PriceSegment quá vô lý)
big_brands = {"APPLE", "SAMSUNG", "OPPO", "XIAOMI", "VIVO", "REALME"}
if {"Brand", "PriceSegment"}.issubset(df_clean.columns):
    mask_weird = df_clean["Brand"].str.upper().isin(big_brands) & df_clean["PriceSegment"].isin(["Low"])
    weird_count = int(mask_weird.sum())
    if weird_count > 0:
        print(f"[WARN] {weird_count} dòng có Brand lớn nhưng PriceSegment=Low — kiểm tra lại quy tắc phân đoạn giá.")

[WARN] 24 dòng có Brand lớn nhưng PriceSegment=Low — kiểm tra lại quy tắc phân đoạn giá.


In [204]:
# mã hoá nhãn giản lược (không one-hot ở đây để tránh phình cột)

DO_MINIMAL_ENCODING = False
if DO_MINIMAL_ENCODING:
    cat_for_codes = ["PriceSegment", "PopularitySegment", "RatingSegment", "BrandCategory", "DuplicateStatus"]
    for c in cat_for_codes:
        if c in df_clean.columns:
            df_clean[c + "Code"] = df_clean[c].astype("category").cat.codes

In [205]:
df_clean = df[columns_keep].copy()

In [209]:
df_clean.to_excel("../data_final/data_final.xlsx", index=False)
print("[INFO] Đã lưu data_final.xlsx với dữ liệu đã làm sạch & dễ hiểu")

[INFO] Đã lưu data_final.xlsx với dữ liệu đã làm sạch & dễ hiểu
